# Faruq-v3 LFDet FTIF breadth screening

Three predeclared arms: **FT1** specific prompt + text-to-image cross-attention, **FT2** base+specific prompt + cross-attention, **FT3** base+specific + LFDet bidirectional alignment. Text encoder is frozen. `OpenAI CLIP ViT-B/32` is an explicit transfer choice because the paper text available to this project does not expose the exact CLIP variant. Prompts are frozen from SNI-21 ontology only. Test is never extracted/opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os,shutil,subprocess,sys,tarfile,time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/lfdet-ftif-text-image-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
cmd=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(cmd)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('clone gagal')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','open_clip_torch'],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt','experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json'))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists()
PRIMARY_OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-lfdet-ftif-screening-v1'
CHUNK3_OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-breadth-screening-batch-v1/candidates/FTIF'
def saved_epochs(root,arm):
    csv_path=root/f'{arm}_seed42/results.csv'
    return max(0,len(csv_path.read_text(errors='replace').splitlines())-1) if csv_path.is_file() else 0
output_progress={root:sum(saved_epochs(root,arm) for arm in ('FT1','FT2','FT3')) for root in (PRIMARY_OUTPUT,CHUNK3_OUTPUT)}
OUTPUT=max(output_progress,key=output_progress.get) if max(output_progress.values())>0 else PRIMARY_OUTPUT
OUTPUT.mkdir(parents=True,exist_ok=True)
cache_candidates=(OUTPUT/'sni21_openai_clip_vit_b32_text_embeddings.pt',PRIMARY_OUTPUT/'sni21_openai_clip_vit_b32_text_embeddings.pt',CHUNK3_OUTPUT/'sni21_openai_clip_vit_b32_text_embeddings.pt')
TEXT_CACHE=next((path for path in cache_candidates if path.is_file()),cache_candidates[0])
print('GPU:',torch.cuda.get_device_name(0))
print('ROOT PROGRESS:',{str(root):{arm:saved_epochs(root,arm) for arm in ('FT1','FT2','FT3')} for root in output_progress})
print('RESUME OUTPUT:',OUTPUT)
print('TEXT CACHE:',TEXT_CACHE)


In [ ]:
from coffee_detector.ftif import generate_clip_text_embeddings
PROMPTS=REPO/'configs/ftif/sni21_prompts.yaml'
if not TEXT_CACHE.is_file():
    meta=generate_clip_text_embeddings(PROMPTS,TEXT_CACHE,model_name='ViT-B-32',pretrained='openai',device='cuda')
    print(meta)
else:
    print('Reuse frozen text cache:',TEXT_CACHE)


In [ ]:
cmd=[sys.executable,'-m','pytest','-q','tests/test_ftif.py']
print('STATIC CHECK:',' '.join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_ftif_screening','--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),'--d0-checkpoint',str(D0),'--text-embeddings',str(TEXT_CACHE),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(cmd),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(p.stdout)
if p.returncode: raise RuntimeError(f'FTIF screen gagal {p.returncode}')


In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/lfdet_ftif_seed42_screening.json'; r=json.loads(SUMMARY.read_text())
assert r['test_opened'] is False and r['test_images_accessed'] is False
rows=[{'model':k,**v} for k,v in r['controls'].items()]+[{'model':k,**v} for k,v in r['candidate'].items()]
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
for arm in ('FT1','FT2','FT3'):
    print(arm,r['decisions'][arm]['decision'],r['decisions'][arm]['delta_vs_D0FT'])
print('TEXT ENCODER:',r['text_encoder']); print('SUMMARY:',SUMMARY)
